# Grade Level Appropriateness Evaluator

**The Grade Level Appropriateness Evaluator** helps you assess whether AI-generated text is suitable for a given grade band. When you run a passage through the evaluator, it returns a structured output that includes:

* **grade_band**: The target grade band where the text can be read independently.
* **alternative_grade_band**: A fallback grade band where the text may still be useful with extra support.
* **scaffolding_needed**: Specific supports (like vocabulary pre-teaching) that make the text accessible outside the target band.
* **reasoning**: Numbered bullet points for the 3 analysis steps, plus a 4th `synthesis` bullet.

This gives you a clear signal about text difficulty and how it can be adapted, so you can generate content that matches learners' needs.

Everything this notebook runs — the model, the temperature, the prompts, the output schema — is loaded from `config.json` in this directory. Nothing is hardcoded below, so the notebook cannot drift from the canonical assets.

In [ ]:
%pip install -qU langchain-google-genai langchain pydantic dotenv

In [ ]:
import getpass
import hashlib
import json
import os
import pprint as pp
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Check for the API key
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

### Load the canonical assets

`config.json` is the source of truth. The prompt files it declares are loaded from disk and
verified against the `sha256` recorded in the config — a drift tripwire. `scripts/check.py`
enforces the same hashes in CI.

In [ ]:
ASSETS_DIR = Path(".")

with open(ASSETS_DIR / "config.json") as f:
    CONFIG = json.load(f)

# Standalone schema files (config.json references them via $ref by path).
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message declared in config, in order. Each message has
# {role, source_path, sha256}; a hash mismatch means the prompt on disk drifted
# from what the config pins, so we fail loudly rather than silently evaluate.
PROMPT_MESSAGES = []  # list of (role, text) tuples, preserving config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    text = (ASSETS_DIR / msg_spec["source_path"]).read_text()
    actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
    declared_sha = msg_spec["sha256"]
    assert actual_sha == declared_sha, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

_STEP = CONFIG["steps"][0]

print(f"Loaded {CONFIG['evaluator']['id']} from {ASSETS_DIR.resolve()}")
print(f"  model:       {_STEP['model']['name']}")
print(f"  temperature: {_STEP['generation']['temperature']}")
print(f"  grade bands: {OUTPUT_SCHEMA['$defs']['GradeBand']['enum']}")
print("  prompts:")
for msg_spec, (role, text) in zip(_STEP["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    print(f"    {role:>6}  {msg_spec['source_path']:<12} ({len(text):>5} chars, sha {sha})")

### Set up the Grade Level Appropriateness Evaluator function

> **Note — Flesch-Kincaid is computed by the model.** `system.txt` embeds the FK formula and asks
> the model to apply it. Every other evaluator in this repo precomputes FK deterministically as a
> declared `preprocessing` step and injects it as `{fk_score}`. GLA is scheduled to move to that
> contract; until then the quantitative signal can vary run to run. See the `TODO` on
> `steps[0].description` in `config.json`.

In [ ]:
def evaluate_grade_level(text: str):
    """
    Evaluate the grade level appropriateness of a text using the canonical config in
    this directory (config.json + system.txt + user.txt).

    Returns a dict with full I/O trace fields:
      - rendered_prompt:  the actual list of messages sent to the model (input-side trace)
      - raw_output:       the AIMessage returned by the LLM (keeps response/usage metadata)
      - raw_text:         just the string content of the AIMessage
      - formatted_output: the parsed dict matching OUTPUT_SCHEMA
      - usage:            token-usage metadata if the provider returned it

    The LLM is invoked ONCE; include_raw=True returns both the raw AIMessage and the
    parsed output without a second call.
    """
    # parser.kind == "structured_output" -> use the model's native output enforcement,
    # driven by output_schema.json rather than a prompt-side format instruction.
    llm = ChatGoogleGenerativeAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    # Every message's content was loaded from disk and hash-verified above, so we can
    # feed the (role, text) tuples straight into the template.
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        inputs = {"text": text}
        rendered_messages = prompt_template.format_messages(**inputs)
        raw = structured_llm.invoke(rendered_messages)

        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output": raw["raw"],
            "raw_text": raw["raw"].content,
            "formatted_output": raw["parsed"],
            "usage": getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

# Try evaluating text to determine appropriate grade level

Evaluator may take up to a minute to run. While the evaluator is running, you will see that the
status of the kernel is busy. For Jupyter, you can see the status of kernel at the bottom of the page.

To evaluate your own text, replace the text and run the cell again.

In [ ]:
# Swap out text here for your own text to evaluate
text = """
Tides are the rise and fall of sea levels caused by the combined effects of the gravitational forces exerted by the Moon and the Sun and the rotation of the Earth.
The times and amplitude of tides at a locale are influenced by the alignment of the Sun and Moon, by the pattern of tides in the deep ocean, by the amphidromic systems of the oceans, and the shape of the coastline and near-shore bathymetry (see Timing). Some shorelines experience a semi-diurnal tide - two nearly equal high and low tides each day. Other locations experience a diurnal tide - only one high and low tide each day. A "mixed tide"; two uneven tides a day, or one high and one low, is also possible.
Tides vary on timescales ranging from hours to years due to a number of factors. To make accurate records, tide gauges at fixed stations measure the water level over time. Gauges ignore variations caused by waves with periods shorter than minutes. These data are compared to the reference (or datum) level usually called mean sea level.
"""

result = evaluate_grade_level(text)

if isinstance(result, dict):
    out = result["formatted_output"]
    print(f"Target grade band:      {out['grade_band']}")
    print(f"Alternative grade band: {out['alternative_grade_band']}")
    print(f"Scaffolding needed:     {out['scaffolding_needed']}")
    print(f"\nReasoning:\n{out['reasoning']}")
else:
    print(result)

### Full I/O trace

In [ ]:
print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
pp.pprint(result["rendered_prompt"])

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(result["raw_text"])

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
pp.pprint(result["formatted_output"])

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(result["usage"])

### Sniff-test runner

Runs the cases in `fixtures.json` and compares the predicted grade bands against the expected
labels. Only the band fields are checked — `reasoning` and `scaffolding_needed` are free text and
vary across runs.

In [ ]:
fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["path"]
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

results = []
for fx in fixtures:
    out = evaluate_grade_level(text=fx["input"]["text"])
    if isinstance(out, str):  # error path
        results.append({"id": fx["id"], "status": "error", "detail": out})
        continue
    predicted = out["formatted_output"]
    mismatches = {
        field: (predicted[field], want)
        for field, want in fx["expected"].items()
        if predicted[field] != want
    }
    results.append({
        "id": fx["id"],
        "status": "pass" if not mismatches else "fail",
        "detail": mismatches or predicted["grade_band"],
    })

print("=" * 78)
print(f"{'ID':>10}  {'STATUS':<8}  DETAIL")
print("=" * 78)
for r in results:
    print(f"{r['id']:>10}  {r['status'].upper():<8}  {r['detail']}")
print("=" * 78)
n_pass = sum(1 for r in results if r["status"] == "pass")
print(f"Summary: {n_pass} pass, {len(results) - n_pass} not passing  --  total {len(results)}")